# 01.6 Losses and Optimizers

This notebook starts answering two core questions in training:

1. how wrong is the model?
2. in which direction should parameters be updated?

These two questions are handled by:

- loss function
- optimizer

## Learning Goals

After this notebook, you should be able to:

1. Understand the role of the loss function.
2. Distinguish common losses for regression and classification.
3. Understand input-output requirements for `MSELoss` and `CrossEntropyLoss`.
4. Understand the optimizer's role in parameter updates.
5. Read the meaning of `zero_grad()`, `backward()`, and `step()`.
6. Prepare for the full training loop.

In [ ]:
import torch
import torch.nn as nn

## What Is a Loss Function?

A loss function turns model error into a number that can be minimized. Lower loss means the model's predictions are closer to the target according to the chosen definition of "closer." Different tasks need different loss functions because regression targets, class labels, and multilabel targets have different meanings.

The loss is also the scalar value that `backward()` starts from during training.

## `MSELoss` for Regression

`MSELoss` is a common loss for regression tasks. It computes prediction error, squares the error so positive and negative mistakes both count, and then averages the squared errors. Squaring also penalizes larger mistakes more strongly than smaller mistakes.

In [ ]:
pred = torch.tensor([[2.5], [0.0], [2.0], [8.0]], dtype=torch.float32)
target = torch.tensor([[3.0], [-0.5], [2.0], [7.0]], dtype=torch.float32)

mse_loss = nn.MSELoss()
loss = mse_loss(pred, target)
print("MSE loss =", loss.item())

In regression, a common habit is to keep `pred` and `target` shapes as consistent as possible.


In [ ]:
# Exercise 1
#
# Use nn.MSELoss to compute the regression loss below.
#
# Steps:
# - Create loss_fn as nn.MSELoss().
# - Call loss_fn(pred, target).
# - Print loss.item() so you see the Python number.

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# Exercise 1 Reference Solution

pred = torch.tensor([[1.0], [2.0], [3.0]])
target = torch.tensor([[1.5], [2.5], [2.0]])
loss_fn = nn.MSELoss()
loss = loss_fn(pred, target)
print(loss.item())

## `CrossEntropyLoss` for Classification

`CrossEntropyLoss` is common for multiclass classification. It expects raw model scores, usually called logits, not probabilities. It also expects targets to be integer class indices with dtype `torch.long`, not one-hot vectors.

A common mistake is to apply softmax before passing outputs to `CrossEntropyLoss`. Do not do that here; the loss function already handles the needed log-softmax calculation internally.

In [ ]:
logits = torch.tensor(
    [
        [2.0, 0.5, -1.0],
        [0.1, 0.2, 2.5],
        [1.5, 1.1, 0.3],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 2, 1], dtype=torch.long)

ce_loss = nn.CrossEntropyLoss()
loss = ce_loss(logits, targets)

print("logits.shape =", logits.shape)
print("targets.shape =", targets.shape)
print("CrossEntropy loss =", loss.item())

Shape requirements:

- `logits.shape == (batch_size, num_classes)`
- `targets.shape == (batch_size,)`

Targets are class indices, not one-hot vectors.


In [ ]:
# Exercise 2
#
# Use nn.CrossEntropyLoss to compute the classification loss below.
#
# Shape meaning:
# - logits.shape == (3, 2), so there are 3 samples and 2 class scores per sample.
# - targets.shape == (3,), so each sample has one integer class label.
#
# Steps:
# - Create loss_fn as nn.CrossEntropyLoss().
# - Call loss_fn(logits, targets).
# - Print loss.item().

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)

# loss_fn =
# loss =
# print(loss.item())

In [ ]:
# Exercise 2 Reference Solution

logits = torch.tensor(
    [
        [1.0, 0.5],
        [0.2, 2.0],
        [1.2, 1.1],
    ],
    dtype=torch.float32,
)
targets = torch.tensor([0, 1, 0], dtype=torch.long)
loss_fn = nn.CrossEntropyLoss()
loss = loss_fn(logits, targets)
print(loss.item())

## Optimizer

The loss tells you how wrong the model is, and the optimizer tells you how to update the parameters.

Common optimizers:

- `SGD`
- `Adam`

You do not need the full formulas yet, but you should know they update parameters based on gradients.


In [ ]:
model = nn.Linear(2, 1)
optimizer_sgd = torch.optim.SGD(model.parameters(), lr=0.1)
optimizer_adam = torch.optim.Adam(model.parameters(), lr=0.01)

print("SGD optimizer / SGD optimizer:")
print(optimizer_sgd)
print()
print("Adam optimizer / Adam optimizer:")
print(optimizer_adam)

## 5. `zero_grad()`、`backward()`、`step()`
## `zero_grad()`, `backward()`, and `step()`

The three most central actions in training are:

1. clear old gradients
2. compute new gradients from the loss
3. update parameters using the gradients

In [ ]:
torch.manual_seed(0)

model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.1)
loss_fn = nn.MSELoss()

x = torch.tensor([[1.0], [2.0], [3.0]])
y = torch.tensor([[2.0], [4.0], [6.0]])

print("parameters before update / parameters before update:")
for name, param in model.named_parameters():
    print(name, param.data)

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print()
print("loss =", loss.item())
print()
print("parameters after update / parameters after update:")
for name, param in model.named_parameters():
    print(name, param.data)

This short block is already very close to a full training loop.


In [ ]:
# Exercise 3
#
# Perform one full parameter update.
#
# Fill in the standard training-step order:
# 1. optimizer.zero_grad()
# 2. pred = model(x)
# 3. loss = loss_fn(pred, y)
# 4. loss.backward()
# 5. optimizer.step()
#
# After the step, print loss.item(). The goal is not to train a good model in
# one step; the goal is to practice the order of operations.

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

# TODO

In [ ]:
# Exercise 3 Reference Solution

torch.manual_seed(1)
model = nn.Linear(1, 1)
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)
loss_fn = nn.MSELoss()
x = torch.tensor([[1.0], [2.0]])
y = torch.tensor([[2.0], [4.0]])

optimizer.zero_grad()
pred = model(x)
loss = loss_fn(pred, y)
loss.backward()
optimizer.step()

print("loss =", loss.item())
for name, param in model.named_parameters():
    print(name, param.data)

## Intuition for `SGD` vs `Adam`

SGD updates parameters in a direct way using the current gradients and the learning rate. Adam keeps additional running statistics of gradients, which often makes it easier to use with default settings and faster to get a reasonable result.

This is not an absolute rule. SGD can work very well with good tuning, and Adam can still fail with a bad learning rate. At this stage, the practical takeaway is that optimizer choice affects how training moves through parameter space.

## Mapping Task Type, Output Layer, and Loss Function

Always choose the output layer and loss function together. For regression, the model often outputs one continuous value and uses `MSELoss` or `L1Loss`. For multiclass classification, the model outputs one logit per class and usually uses `CrossEntropyLoss` with integer class labels. For multilabel classification, the model outputs one logit per label and often uses `BCEWithLogitsLoss`.

If the output shape and target meaning do not match the loss function, training will either crash or optimize the wrong objective.

In [ ]:
# Exercise 4
#
# Decide which loss function fits each scenario.
#
# Scenario:
# - predict tomorrow's temperature
#
# Write one or two full sentences. Your answer should mention that temperature
# is a continuous target, so this is a regression problem.

Exercise 4 Reference Answer

1. Predict tomorrow's temperature: regression, so `MSELoss` or `L1Loss` is a natural fit.
2. Choose one class among several exclusive classes: multiclass classification, so use `CrossEntropyLoss` with logits of shape `(batch, num_classes)` and integer class labels.
3. Predict several independent yes/no labels at once: multilabel classification, so use `BCEWithLogitsLoss` with one logit per label.

Loss choice depends on the target meaning and expected output shape.

## Summary

The training chain in this notebook is the foundation of supervised learning in PyTorch. The model produces outputs, the loss compares those outputs with targets, backpropagation computes gradients, and the optimizer updates parameters using those gradients.

Before moving on, make sure you can explain why regression and classification use different losses, why `CrossEntropyLoss` expects logits and class-index targets, and why the optimizer step must happen after gradients are computed.